In [1]:
import os
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED

In [4]:
import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

In [5]:
topicnet.__file__

! ls /home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [6]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [7]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [8]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/RTL_Wiki_person.csv',
)

dataset.get_possible_modalities()

{'@bigram', '@lemmatized'}

In [9]:
dataset._internals_folder_path

'/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/RTL_Wiki_person__internals'

In [10]:
MAIN_MODALITY = '@lemmatized'

In [11]:
dataset._data.head()

,Unnamed: 0,id,raw_text,vw_text
id,,,,
İsmet_İnönü,0,İsmet_İnönü,Mustafa İsmet İnönü (September 24 1884 – Decem...,İsmet_İnönü |@lemmatized mustafa:2 smet:7 nönü...
Clara_Petacci,1,Clara_Petacci,Clara Petacci (Claretta Petacci) (28 February ...,Clara_Petacci |@lemmatized clara:5 petacci:15 ...
Jack_Ruby,2,Jack_Ruby,"Jacob Rubenstein (March 25, 1911 – January 3, ...",Jack_Ruby |@lemmatized jacob:2 rubenstein:5 ma...
Knud_Rasmussen,3,Knud_Rasmussen,"Knud Johan Victor Rasmussen (June 7, 1879–Dece...",Knud_Rasmussen |@lemmatized knud:15 johan:3 vi...
Gerald_Schroeder,4,Gerald_Schroeder,"Gerald L. Schroeder is a scientist, author, an...",Gerald_Schroeder |@lemmatized gerald:4 l:1 sch...


In [12]:
dataset._data.shape

(1201, 4)

In [13]:
dataset.get_dictionary()

artm.Dictionary(name=00f261bc-a5d2-4e7a-86d5-a0e9ceff04f9, num_entries=124241)

In [14]:
dictionary = dataset.get_dictionary()

In [15]:
print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=00f261bc-a5d2-4e7a-86d5-a0e9ceff04f9, num_entries=124241)


In [16]:
dictionary.filter(min_df=2, max_df_rate=0.5)

artm.Dictionary(name=00f261bc-a5d2-4e7a-86d5-a0e9ceff04f9, num_entries=37739)

In [17]:
dataset._cached_dict = dictionary

In [18]:
dataset.get_dictionary()

artm.Dictionary(name=00f261bc-a5d2-4e7a-86d5-a0e9ceff04f9, num_entries=37739)

In [19]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [20]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 7.74 s, sys: 330 ms, total: 8.07 s
Wall time: 7.98 s


In [21]:
co_occurences.shape

(37739, 37739)

In [22]:
dataset.get_dictionary()

artm.Dictionary(name=00f261bc-a5d2-4e7a-86d5-a0e9ceff04f9, num_entries=37739)

In [23]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [24]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [25]:
KnownModel

<enum 'KnownModel'>

In [26]:
PARAMS_EXPLORED

{<KnownModel.LDA: 'LDA'>: {'prior': ['symmetric', 'asymmetric', 'heuristic']},
 <KnownModel.PLSA: 'PLSA'>: {},
 <KnownModel.TLESS: 'TARTM'>: {},
 <KnownModel.SPARSE: 'sparse'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1]},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': [0.02,
   0.05,
   0.1]},
 <KnownModel.ARTM: 'ARTM'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1],
  'decorrelation_tau': [0.02, 0.05, 0.1]}}

In [27]:
NUM_TOPICS = 50  # vary
NUM_TRAINS = 3
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

In [28]:
dataset.get_dictionary()

artm.Dictionary(name=00f261bc-a5d2-4e7a-86d5-a0e9ceff04f9, num_entries=37739)

In [29]:
dictionary = dataset.get_dictionary()

## Test

In [30]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [54]:
%%time

model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=1,
)

model._fit(dataset.get_batch_vectorizer(), num_iterations=10)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


CPU times: user 3min 1s, sys: 2.88 s, total: 3min 4s
Wall time: 1min 16s


In [55]:
list(model.scores.keys())

['PerplexityScore@all',
 'SparsityThetaScore',
 'SparsityPhiScore@lemmatized',
 'PerplexityScore@lemmatized',
 'TopicKernel@lemmatized.average_coherence',
 'TopicKernel@lemmatized.average_contrast',
 'TopicKernel@lemmatized.average_purity',
 'TopicKernel@lemmatized.average_size',
 'TopicKernel@lemmatized.coherence',
 'TopicKernel@lemmatized.contrast',
 'TopicKernel@lemmatized.purity',
 'TopicKernel@lemmatized.size',
 'TopicKernel@lemmatized.tokens']

In [58]:
model.scores[f'PerplexityScore{MAIN_MODALITY}']

[60934.58203125,
 9283.0244140625,
 8148.603515625,
 6405.390625,
 5430.90283203125,
 4980.9423828125,
 4732.5595703125,
 4578.2822265625,
 4477.1103515625,
 4408.48046875]

In [59]:
phi = model.get_phi()
target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
target_topic_names = [phi.columns[i] for i in target_topic_indices]

custom_scores = [
    TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )
    for top in [20]  # [10, 20, 50, 100]
]
custom_scores = custom_scores + [
    DiversityScore(
        name=f'diversity_{metric}',
        topic_names=['topic_0', 'topic_1'],
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

for score in custom_scores:
    res = score.call(model)

    print(score._name)
    print(res)

    if isinstance(score, TopTokenCoherence):
        res_by_topic = score.call_by_topic(model)

        print(res_by_topic)

coherence_20
[0.0273816]
{0: array([0.]), 1: array([0.02958778]), 2: array([0.04763114]), 3: array([0.01888187]), 4: array([0.]), 5: array([0.03611333]), 6: array([0.]), 7: array([0.09164026]), 8: array([0.04763114]), 9: array([0.]), 10: array([0.]), 11: array([0.07011646]), 12: array([0.]), 13: array([0.]), 14: array([0.]), 15: array([0.]), 16: array([0.]), 17: array([0.08207206]), 18: array([0.02869565]), 19: array([0.09526229])}
diversity_euclidean
0.038674582514403096
diversity_jensenshannon
0.038674582514403096
diversity_hellinger
0.038674582514403096
diversity_cosine
0.038674582514403096


In [60]:
model.get_phi(class_ids=MAIN_MODALITY)['topic_18'].sort_values(ascending=False)

modality     token          
@lemmatized  вид                0.011582
             птица              0.011062
             территория         0.006171
             район              0.005858
             река               0.005784
                                  ...   
             минималистичный    0.000000
             натурщица          0.000000
             афрасиябнуть       0.000000
             nadh               0.000000
             рлэ                0.000000
Name: topic_18, Length: 61688, dtype: float32

In [61]:
model.class_ids

{'@lemmatized': 1}

In [62]:
KNOWN_METRICS

['euclidean', 'jensenshannon', 'hellinger', 'cosine']

In [30]:
MAIN_MODALITY

'@lemmatized'

In [31]:
def fit_and_compute_scores(model, dataset):
    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [32]:
co_occurences.shape

(37739, 37739)

In [33]:
BEST_PARAMS = dict()

## PLSA

In [35]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [36]:
NUM_TOPICS

50

In [51]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [37]:
BEST_PARAMS[KnownModel.PLSA] = None

## Sparse

In [38]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [54]:
results = dict()

for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['sparse_sp_tau']:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (sparse_sp_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.SPARSE,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={'sparse_sp_tau': sparse_sp_tau, 'smooth_bcg_tau': smooth_bcg_tau}
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
 
            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(-0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.1782283021472972
smooth_theta_bcg: 68.4464262237609
sparse_phi_sp: -0.09853889938285393
sparse_theta_sp: -3.096385948217755
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.1782283021472972
smooth_theta_bcg: 68.4464262237609
sparse_phi_sp: -0.09853889938285393
sparse_theta_sp: -3.096385948217755
2


...

smooth_phi_bcg: 4.59848197119985
smooth_theta_bcg: 144.49801091682855
sparse_phi_sp: -0.18811971700363023
sparse_theta_sp: -5.91128226477935
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.59848197119985
smooth_theta_bcg: 144.49801091682855
sparse_phi_sp: -0.18811971700363023
sparse_theta_sp: -5.91128226477935
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.59848197119985
smooth_theta_bcg: 144.49801091682855
sparse_phi_sp: -0.18811971700363023
sparse_theta_sp: -5.91128226477935




In [55]:
results

{(-0.05,
  0.05): [{'scores': {'perplexity': 4037.35791015625,
    'coherence_20': array([0.62106338]),
    'diversity_euclidean': 0.053662157922497795,
    'diversity_jensenshannon': 0.6518637276087699,
    'diversity_hellinger': 0.7577147252233769,
    'diversity_cosine': 0.8139776651740249},
   'topic_coherences': {0: 0.5225908625149935,
    1: 0.5331450869041109,
    2: 1.223804867833731,
    3: 0.5261857613384362,
    4: 0.39208000087707134,
    5: 0.5656146718404065,
    6: 0.4665352280865459,
    7: 0.5484668820539617,
    8: 0.5104048141396167,
    9: 0.4572070516716446,
    10: 0.5524335348372929,
    11: 0.5606666346198073,
    12: 0.6096816974914626,
    13: 0.9103697786572178,
    14: 0.6757981901008699,
    15: 1.0258876005948325,
    16: 0.4681103962059163,
    17: 0.5962910863766723,
    18: 0.5783138756166648,
    19: 0.6976794919725269}}, {'scores': {'perplexity': 3993.8310546875,
    'coherence_20': array([0.65199031]),
    'diversity_euclidean': 0.05221421493651865,


In [57]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

(-0.05, 0.05) 3997.9117024739585
(-0.05, 0.1) 4192.648274739583
(-0.1, 0.05) 4168.622721354167
(-0.1, 0.1) 4364.895670572917


In [79]:
# Best: (-0.05, 0.05) 3997.9117024739585

In [39]:
BEST_PARAMS[KnownModel.SPARSE] = {
    'sparse_sp_tau': -0.05,
    'smooth_bcg_tau': 0.05,
}

In [59]:
model.get_phi()['topic_15'].sort_values(ascending=False)

modality     token      
@lemmatized  president      0.010302
             soviet         0.010294
             u              0.007399
             party          0.007391
             union          0.006480
                              ...   
             barre          0.000000
             carcanet       0.000000
             esther         0.000000
             constituent    0.000000
             glorious       0.000000
Name: topic_15, Length: 37739, dtype: float32

## Decorrelation

In [40]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [41]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [62]:
DECORRELATION_TAUS = [0.01] + PARAMS_EXPLORED[KnownModel.DECORRELATION]['decorrelation_tau']

In [63]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (decorrelation_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': decorrelation_tau,
                    'smooth_bcg_tau': smooth_bcg_tau,
                    'sparse_sp_tau': 0.0,
                }
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")

            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(0.01, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.1782283021472972
smooth_theta_bcg: 68.4464262237609
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.1782283021472972
smooth_theta_bcg: 68.4464262237609
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
2


...

smooth_phi_bcg: 4.59848197119985
smooth_theta_bcg: 144.49801091682855
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.59848197119985
smooth_theta_bcg: 144.49801091682855
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.59848197119985
smooth_theta_bcg: 144.49801091682855
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1




In [67]:
len(results)

8

In [68]:
results

{(0.01,
  0.05): [{'scores': {'perplexity': 3685.59716796875,
    'coherence_20': array([0.61894336]),
    'diversity_euclidean': 0.04640070393004205,
    'diversity_jensenshannon': 0.6186979264807327,
    'diversity_hellinger': 0.7111349426386786,
    'diversity_cosine': 0.7872713557024492},
   'topic_coherences': {0: 0.5296769235033192,
    1: 0.5092230844790822,
    2: 1.1477835075690603,
    3: 0.4384294633128111,
    4: 0.3745164202602821,
    5: 0.5763722643248919,
    6: 0.4066091707637462,
    7: 0.4659966325926779,
    8: 0.646370576242691,
    9: 0.44652065121310996,
    10: 0.5239734581459495,
    11: 0.640919067764959,
    12: 0.7276167743004693,
    13: 0.9842329500460499,
    14: 0.5923111223021188,
    15: 0.9694065417341837,
    16: 0.5243207578700126,
    17: 0.6269395014186195,
    18: 0.5499688279863997,
    19: 0.6976794919725269}}, {'scores': {'perplexity': 3653.32421875,
    'coherence_20': array([0.63644294]),
    'diversity_euclidean': 0.04577241320961831,
    '

In [69]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    print(k, mean_ppl)

(0.01, 0.05) 3651.9127604166665
(0.01, 0.1) 3808.7379557291665
(0.02, 0.05) 3652.208740234375
(0.02, 0.1) 3809.4609375
(0.05, 0.05) 3672.995361328125
(0.05, 0.1) 3828.4727376302085
(0.1, 0.05) 3806.1436360677085
(0.1, 0.1) 3943.1307779947915


In [ ]:
#  Best:               (0.01, 0.05) 3651.9127604166665
# Close (very close): (0.02, 0.05) 3652.208740234375

In [42]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.02,
    'smooth_bcg_tau': 0.05,
}

## ARTM

In [72]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [73]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [74]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [75]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.ARTM]['sparse_sp_tau']:
        for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
            key = (decorrelation_tau, sparse_sp_tau, smooth_bcg_tau)
            results[key] = []
    
            print(key)
    
            for seed in range(NUM_TRAINS):
                print(seed)
                
                model = init_model_from_family(
                    family=KnownModel.ARTM,
                    dataset=dataset,
                    main_modality=MAIN_MODALITY,
                    num_topics=NUM_TOPICS,
                    seed=seed,
                    model_params={
                        'decorrelation_tau': decorrelation_tau,
                        'smooth_bcg_tau': smooth_bcg_tau,
                        'sparse_sp_tau': sparse_sp_tau,
                    }
                )
    
                for reg in model.regularizers.data:
                    print(f"{reg}: {model.regularizers[reg].tau}")
    
                scores = fit_and_compute_scores(model, dataset)
                results[key].append(scores)

            print()

        print()

    print()

(0.01, -0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.1782283021472972
smooth_theta_bcg: 68.4464262237609
sparse_phi_sp: -0.09853889938285393
sparse_theta_sp: -3.096385948217755
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.1782283021472972
smooth_theta_bcg: 68.4464262237609
sparse_phi_sp: -0.09853889938285393
sparse_theta_sp: -3.096385948217755
decorrelation: 0.01
2


...

smooth_phi_bcg: 4.59848197119985
smooth_theta_bcg: 144.49801091682855
sparse_phi_sp: -0.18811971700363023
sparse_theta_sp: -5.91128226477935
decorrelation: 0.1
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.59848197119985
smooth_theta_bcg: 144.49801091682855
sparse_phi_sp: -0.18811971700363023
sparse_theta_sp: -5.91128226477935
decorrelation: 0.1
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.59848197119985
smooth_theta_bcg: 144.49801091682855
sparse_phi_sp: -0.18811971700363023
sparse_theta_sp: -5.91128226477935
decorrelation: 0.1





In [79]:
len(results)

16

In [80]:
ppls = []

for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    ppls.append(mean_ppl)

    print(k, mean_ppl)

(0.01, -0.05, 0.05) 4005.2604166666665
(0.01, -0.05, 0.1) 4197.793294270833
(0.01, -0.1, 0.05) 4175.214518229167
(0.01, -0.1, 0.1) 4370.04541015625
(0.02, -0.05, 0.05) 4013.9186197916665
(0.02, -0.05, 0.1) 4204.282063802083
(0.02, -0.1, 0.05) 4183.7529296875
(0.02, -0.1, 0.1) 4376.525065104167
(0.05, -0.05, 0.05) 4050.0513509114585
(0.05, -0.05, 0.1) 4231.96240234375
(0.05, -0.1, 0.05) 4217.027180989583
(0.05, -0.1, 0.1) 4404.224934895833
(0.1, -0.05, 0.05) 4152.126139322917
(0.1, -0.05, 0.1) 4326.097981770833
(0.1, -0.1, 0.05) 4325.7666015625
(0.1, -0.1, 0.1) 4503.3408203125


In [78]:
sorted(ppls)[:5]

[4005.2604166666665,
 4013.9186197916665,
 4050.0513509114585,
 4152.126139322917,
 4175.214518229167]

In [ ]:
#  Best: (0.01, -0.05, 0.05) 4005.2604166666665
# Close: (0.02, -0.05, 0.05) 4013.9186197916665

In [43]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.01,
    'sparse_sp_tau':    -0.05,
    'smooth_bcg_tau':    0.05,
}

## TLESS

In [83]:
PARAMS_EXPLORED[KnownModel.TLESS]

{}

In [84]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.TLESS,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [85]:
results

[{'scores': {'perplexity': 4470.26220703125,
   'coherence_20': array([0.50350006]),
   'diversity_euclidean': 0.07339134695990905,
   'diversity_jensenshannon': 0.7507417763523968,
   'diversity_hellinger': 0.8797425961824018,
   'diversity_cosine': 0.9194666559989366},
  'topic_coherences': {0: 0.41787068406895206,
   1: 0.456275859995534,
   2: 0.9470673960465779,
   3: 0.4323759825588596,
   4: 0.43040937839197607,
   5: 0.6372741566323233,
   6: 0.40098452275557606,
   7: 0.5116468917214952,
   8: 0.3951207016098807,
   9: 0.34156842448508534,
   10: 0.4302915906964934,
   11: 0.577606283095257,
   12: 0.5127387869840155,
   13: 0.4330635816825909,
   14: 0.4147440052247522,
   15: 0.4369039486307174,
   16: 0.49580618921812264,
   17: 0.6535039034167227,
   18: 0.4948559451083293,
   19: 0.6498928923686695}},
 {'scores': {'perplexity': 4426.88134765625,
   'coherence_20': array([0.51372409]),
   'diversity_euclidean': 0.07514543703257033,
   'diversity_jensenshannon': 0.753548245

In [86]:
# Best:

In [44]:
BEST_PARAMS[KnownModel.TLESS] = None

## LDA

In [88]:
PARAMS_EXPLORED[KnownModel.LDA]

{'prior': ['symmetric', 'asymmetric', 'heuristic']}

In [89]:
results = dict()

for prior in PARAMS_EXPLORED[KnownModel.LDA]['prior']:
    key = prior
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.LDA,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={'prior': prior}
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        scores = fit_and_compute_scores(model, dataset)
        results[key].append(scores)

    print()

symmetric
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
2


...

smooth_phi: 0.01
smooth_theta: 2.5
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5



In [90]:
results

{'symmetric': [{'scores': {'perplexity': 3572.5078125,
    'coherence_20': array([0.58121574]),
    'diversity_euclidean': 0.04020684921004752,
    'diversity_jensenshannon': 0.5704552078621047,
    'diversity_hellinger': 0.6309496905619789,
    'diversity_cosine': 0.734543445839627},
   'topic_coherences': {0: 0.5541849991659782,
    1: 0.5871515888066066,
    2: 1.14778350756906,
    3: 0.6887570594579875,
    4: 0.5030238762049593,
    5: 0.4626049954826465,
    6: 0.5952980581802646,
    7: 0.3917482195376217,
    8: 0.5072870187235307,
    9: 0.47018409595519345,
    10: 0.4708908734378012,
    11: 0.5999949594614384,
    12: 0.48628548352487966,
    13: 0.670824347735373,
    14: 0.38170479601457663,
    15: 0.8942909692482227,
    16: 0.4445982765295672,
    17: 0.6269628092205252,
    18: 0.5360314229696022,
    19: 0.6047074656399117}},
  {'scores': {'perplexity': 3509.46142578125,
    'coherence_20': array([0.63639777]),
    'diversity_euclidean': 0.0404753674648023,
    'div

In [91]:
ppls = []

for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    ppls.append(mean_ppl)

    print(k, mean_ppl)

symmetric 3531.2176106770835
asymmetric 3529.6451009114585
heuristic 3607.2461751302085


In [92]:
sorted(ppls)

[3529.6451009114585, 3531.2176106770835, 3607.2461751302085]

In [94]:
# Best:               asymmetric 3529.6451009114585
# Close (very close): symmetric 3531.2176106770835

In [45]:
BEST_PARAMS[KnownModel.LDA] = {
    'prior': 'symmetric',
}

In [96]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [46]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [34]:
BEST_PARAMS = {KnownModel.PLSA: None,
 KnownModel.SPARSE: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 KnownModel.DECORRELATION: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 KnownModel.TLESS: None,
 KnownModel.LDA: {'prior': 'symmetric'}}

In [35]:
import json
import warnings

warnings.simplefilter('ignore', UserWarning)

In [36]:
NUM_TRAINS = 20  # 100
COHERENCES = list()

In [37]:
! ls results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [38]:
SAVE_FOLDER = 'results50/rtlwikiperson'

! mkdir -p $SAVE_FOLDER

In [39]:
! ls results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [40]:
! ls results50

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [41]:
# PLSA

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    COHERENCES.extend(
        list(results['topic_coherences'].values())
    )

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [42]:
scores[0]

{'scores': {'perplexity': 2629.09326171875,
  'coherence_20': array([0.65589822]),
  'diversity_euclidean': 0.05163600902877945,
  'diversity_jensenshannon': 0.6252680449182312,
  'diversity_hellinger': 0.7238801758459893,
  'diversity_cosine': 0.8076587889009033},
 'topic_coherences': {0: 0.6523147330347747,
  1: 0.5619361209431922,
  2: 1.1312754478527065,
  3: 0.5987470028548232,
  4: 0.577539253596847,
  5: 0.9937087117189723,
  6: 0.4006058216224602,
  7: 0.42120928672113034,
  8: 0.6563654865303324,
  9: 0.6421566909966528,
  10: 0.7962782033620498,
  11: 0.5418313963977043,
  12: 0.5534567686351402,
  13: 1.0835000251586984,
  14: 0.46495187882647876,
  15: 0.4510567193093095,
  16: 0.6750323297806261,
  17: 0.5359958445523142,
  18: 0.5577451112794043,
  19: 0.7938170466762348,
  20: 0.6855795009399105,
  21: 0.5733253223499296,
  22: 0.37265760452873675,
  23: 0.5072829949180846,
  24: 0.7160720421583341,
  25: 0.9598913000835493,
  26: 0.45745567380183233,
  27: 0.68994760785

In [43]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [57]:
scores

[{'perplexity': 2629.09326171875,
  'coherence_20': 0.6558982238157229,
  'diversity_euclidean': 0.05163600902877945,
  'diversity_jensenshannon': 0.6252680449182312,
  'diversity_hellinger': 0.7238801758459893,
  'diversity_cosine': 0.8076587889009033},
 {'perplexity': 2646.99560546875,
  'coherence_20': 0.6210707709675107,
  'diversity_euclidean': 0.05075327431085609,
  'diversity_jensenshannon': 0.6260045077582149,
  'diversity_hellinger': 0.7249915668514149,
  'diversity_cosine': 0.7997033795504188},
 {'perplexity': 2620.472412109375,
  'coherence_20': 0.6647144084864414,
  'diversity_euclidean': 0.05103858156022477,
  'diversity_jensenshannon': 0.6246642668474883,
  'diversity_hellinger': 0.7229579182817749,
  'diversity_cosine': 0.802086464401499},
 {'perplexity': 2622.9609375,
  'coherence_20': 0.6477534785962323,
  'diversity_euclidean': 0.05154897377870552,
  'diversity_jensenshannon': 0.6236441273612762,
  'diversity_hellinger': 0.7220900421230986,
  'diversity_cosine': 0.799

In [58]:
SAVE_FOLDER

'results50/rtlwikiperson'

In [44]:
with open(SAVE_FOLDER + '/plsa_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [60]:
len(COHERENCES)

1000

In [61]:
COHERENCES[:10]

[0.6523147330347747,
 0.5619361209431922,
 1.1312754478527065,
 0.5987470028548232,
 0.577539253596847,
 0.9937087117189723,
 0.4006058216224602,
 0.42120928672113034,
 0.6563654865303324,
 0.6421566909966528]

In [62]:
COHERENCES[:20]

[0.6523147330347747,
 0.5619361209431922,
 1.1312754478527065,
 0.5987470028548232,
 0.577539253596847,
 0.9937087117189723,
 0.4006058216224602,
 0.42120928672113034,
 0.6563654865303324,
 0.6421566909966528,
 0.7962782033620498,
 0.5418313963977043,
 0.5534567686351402,
 1.0835000251586984,
 0.46495187882647876,
 0.4510567193093095,
 0.6750323297806261,
 0.5359958445523142,
 0.5577451112794043,
 0.7938170466762348]

In [45]:
# Sparse

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.SPARSE,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
        model_params=BEST_PARAMS[KnownModel.SPARSE],
    )

    if seed == 0:
        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    COHERENCES.extend(
        list(results['topic_coherences'].values())
    )

0 smooth_phi_bcg: 2.1782283021472972
smooth_theta_bcg: 68.4464262237609
sparse_phi_sp: -0.039415559753141566
sparse_theta_sp: -1.2385543792871019
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [46]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [47]:
with open(SAVE_FOLDER + '/sparse_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [66]:
len(COHERENCES)

2000

In [67]:
COHERENCES[-10:]

[0.6505673851113477,
 0.5268626638618532,
 0.5617103430407168,
 0.8003218041355288,
 0.5704199798592986,
 1.8088704503945057,
 1.0608758864417727,
 0.8414015342010872,
 1.0119796081601027,
 0.5295188012089266]

In [68]:
max(COHERENCES)

2.3949827273281423

In [69]:
min(COHERENCES)

0.322401940488756

In [48]:
def train_many(model_family, save_file_path):
    scores = []

    # for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
    for seed in range(NUM_TRAINS):
        if seed != NUM_TRAINS - 1:
            print(seed, end=' ')
        else:
            print(seed)
    
        model = init_model_from_family(
            family=model_family,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params=BEST_PARAMS[model_family],
        )
    
        if seed == 0:
            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
    
        results = fit_and_compute_scores(model, dataset)
        # scores.append(results['scores'])
        scores.append(results)
    
        COHERENCES.extend(
            list(results['topic_coherences'].values())
        )

    # for s in scores:
    #     s['coherence_20'] = float(s['coherence_20'])
    
    for s in scores:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

    with open(save_file_path, 'w') as f:
        f.write(
            json.dumps(scores, indent=4)
        )

In [49]:
train_many(KnownModel.DECORRELATION, SAVE_FOLDER + '/decorrelation_with_cohs.json')

0 decorrelation: 0.01
smooth_phi_bcg: 2.1782283021472972
smooth_theta_bcg: 68.4464262237609
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [72]:
len(COHERENCES)

3000

In [50]:
train_many(KnownModel.TLESS, SAVE_FOLDER + '/tless_with_cohs.json')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [74]:
len(COHERENCES)

4000

In [51]:
train_many(KnownModel.LDA, SAVE_FOLDER + '/lda_with_cohs.json')

0 smooth_phi: 0.02
smooth_theta: 0.02
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [53]:
len(COHERENCES)

5000

In [77]:
COHERENCES

[0.6523147330347747,
 0.5619361209431922,
 1.1312754478527065,
 0.5987470028548232,
 0.577539253596847,
 0.9937087117189723,
 0.4006058216224602,
 0.42120928672113034,
 0.6563654865303324,
 0.6421566909966528,
 0.7962782033620498,
 0.5418313963977043,
 0.5534567686351402,
 1.0835000251586984,
 0.46495187882647876,
 0.4510567193093095,
 0.6750323297806261,
 0.5359958445523142,
 0.5577451112794043,
 0.7938170466762348,
 0.6855795009399105,
 0.5733253223499296,
 0.37265760452873675,
 0.5072829949180846,
 0.7160720421583341,
 0.9598913000835493,
 0.45745567380183233,
 0.6899476078517598,
 0.5203201589968638,
 0.808418937345942,
 0.6866323856242342,
 0.4981535582164922,
 1.0439629996059567,
 0.4238377072583979,
 0.5450408263087342,
 1.088445890827498,
 0.5692103743678916,
 0.47459575990836445,
 0.589219164181861,
 0.5666451033722811,
 0.7299191506831981,
 0.6866487478001194,
 0.5007130879137143,
 0.39615459319131846,
 0.4324578711033445,
 1.8013059941781564,
 0.6950432614291051,
 0.48472954

In [88]:
len(COHERENCES)

5000

In [85]:
for p in range(5, 100, 5):
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 5: 0.4036236822746325
10: 0.43878312987320234
15: 0.46049818059903014
20: 0.4818845486539586
25: 0.5013317324596036
30: 0.5222196856513092
35: 0.5436083934788418
40: 0.5633195043276958
45: 0.5818599054363903
50: 0.6026452997823235
55: 0.626103369567515
60: 0.6522721776599065
65: 0.6786401509612466
70: 0.7090512937126278
75: 0.746500738807341
80: 0.7938587462014927
85: 0.8523221339178352
90: 0.9594689208755898
95: 1.1364611976616275


In [86]:
min(COHERENCES), max(COHERENCES)

(0.16986171869471386, 2.3949827273281428)

In [87]:
np.argmin(COHERENCES), np.argmax(COHERENCES)

(2449, 2224)

In [83]:
for p in [2, 98]:
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 2: 0.36355856027194317
98: 1.4506404115493394


In [134]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [82]:
dataset.get_dictionary()

artm.Dictionary(name=fe2d9fcc-1b9a-4605-9521-0dea9e322c3c, num_entries=37739)